<a href="https://colab.research.google.com/github/tk-cry-pto/crypto-bot/blob/main/bot_main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
API_KEY = "QdFwudJeyraSqgmzeAH4DV"
API_SECRET = "ZGbg/ks5kuYRT0mr8CdY3pxwZQ9d+R2Q27eI9qnusSE="

In [11]:
import requests
import time
import pandas as pd
from datetime import datetime

url = "https://api.bitflyer.com/v1/ticker"

data_list = []

position = 0
entry_price = 0

start_time = time.time()

while True:
    try:
        response = requests.get(url, params={"product_code": "BTC_JPY"})
        data = response.json()

        now = datetime.now()
        price = data["ltp"]

        data_list.append({
            "datetime": now,
            "close": price
        })

        df = pd.DataFrame(data_list)
        # df.to_csv("/content/drive/MyDrive/crypto_data/realtime.csv", index=False)

        df["ema20"] = df["close"].ewm(span=20, adjust=False).mean()
        df["ema50"] = df["close"].ewm(span=50, adjust=False).mean()

        signal = "待機"

        if len(df) > 10:
            ema20 = df["ema20"].iloc[-1]
            ema50 = df["ema50"].iloc[-1]
            price_now = df["close"].iloc[-1]

            trend_up = ema20 > ema50
            pullback = abs(price_now - ema20) / price_now < 0.003
            bullish = (
                price_now > df["close"].iloc[-2]
                and price_now > df["close"].iloc[-3]
            )
            trend_strength = abs(ema20 - ema50) / price_now
            time_filter = True

            print(f"trend_up={trend_up}, pullback={pullback}, bullish={bullish}, strength={trend_strength:.4f}, time_filter={time_filter}")

            if trend_up and pullback and bullish and trend_strength > 0.0015 and time_filter:
                signal = "買い"

        # =========================
        # 仮想売買のみ：実注文なし
        # =========================
        if signal == "買い" and position == 0:
            position = 1
            entry_price = price

            print("=================================")
            print("▶ 仮想買い")
            print(f"買い価格: {entry_price}")
            print("=================================")

        elif position == 1:
            profit = (price - entry_price) / entry_price * 100

            exit_reason = None

            if profit > 0.3:
                exit_reason = "仮想利確"

            elif profit < -0.2:
                exit_reason = "仮想損切り"

            if exit_reason:
                print("=================================")
                print(f"■ 仮想売り（{exit_reason}）")
                print(f"売り価格: {price}")
                print(f"エントリー: {entry_price}")
                print(f"損益: {profit:.3f}%")
                print("=================================")

                position = 0
                entry_price = 0

        print(
            f"{now.strftime('%H:%M:%S')} | "
            f"価格: {price:.0f} | "
            f"シグナル: {signal} | "
            f"ポジション: {position}"
        )

        time.sleep(60)

        if time.time() - start_time > 60 * 60 * 24:
            print("24時間経過で終了")
            break

    except KeyboardInterrupt:
        print("停止しました")
        break

    except Exception as e:
        print("エラー:", e)
        time.sleep(10)

07:49:16 | 価格: 12328007 | シグナル: 待機 | ポジション: 0
停止しました
